# Kaggle T4: WELFake + LIAR (Text) — Final Run

This notebook is designed to run end-to-end on Kaggle with GPU enabled (T4).
It writes **CSV tables**, **PNG figures**, and a **zip** of all outputs.

## What it runs
- Two datasets: **WELFake** and **LIAR**
- Text pipeline: **TF-IDF → TruncatedSVD (max_qubits dims) → scaling**
- Qubit scaling: **4, 6, 8, 10** qubits
- Per qubit: classical baselines on the *same reduced representation* + quantum kernel Pegasos
- Noise sweep (at max qubits=10): **0.1%, 0.5%, 1%, 2%, 5%**

## Requirements
1) Upload the **code** (this repo folder containing `new_research/qmlfakenews`) as a Kaggle Dataset, or clone it into `/kaggle/working`.
2) Upload **WELFake_Dataset.csv** and the **liar-dataset/** folder as Kaggle Datasets.

In [ ]:
# Cell 1/3 — install dependencies (robust fallback)
import sys, subprocess, textwrap

def pip_install(pkgs):
    cmd = [sys.executable, '-m', 'pip', 'install', '-U', '-q'] + list(pkgs)
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

# Pinned-first strategy: try the versions we already validated locally;
# if Kaggle can't resolve them, fall back to compatible unpinned installs.
pinned = [
    'numpy>=1.26',
    'pandas>=2.2',
    'scikit-learn>=1.5',
    'matplotlib>=3.9',
    'seaborn>=0.13',
    'tqdm>=4.66',
    'python-docx>=1.1',
    # Quantum stack (pinned to the versions used in this project locally)
    'qiskit==2.4.1',
    'qiskit-aer==0.17.2',
    'qiskit-machine-learning>=0.8'
]
fallback = [
    'numpy', 'pandas', 'scikit-learn', 'matplotlib', 'seaborn', 'tqdm', 'python-docx',
    'qiskit', 'qiskit-aer', 'qiskit-machine-learning'
]

try:
    pip_install(pinned)
except Exception as e:
    print('Pinned install failed; falling back to unpinned. Error:', repr(e))
    pip_install(fallback)

import numpy as np
import pandas as pd
import sklearn
import matplotlib
import qiskit

print('Versions:')
print('  python', sys.version.split()[0])
print('  numpy', np.__version__)
print('  pandas', pd.__version__)
print('  sklearn', sklearn.__version__)
print('  matplotlib', matplotlib.__version__)
print('  qiskit', getattr(qiskit, '__version__', 'unknown'))

In [ ]:
# Cell 2/3 — minimal project helper stubs (AUTOINLINE)
# These provide the minimal interfaces required by Cell 3/3 so the notebook can run
# You can replace these with the full qmlfakenews module code later.

import numpy as np
import pandas as pd
from typing import List, Dict, Any
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

def run_classical_baselines(X_train, y_train, X_test, y_test) -> List[Dict[str,Any]]:
    """Fit a few simple classical baselines and return results as a list of dicts."""
    results = []
    try:
        lr = LogisticRegression(max_iter=1000, random_state=0).fit(X_train, y_train)
        preds = lr.predict(X_test)
        results.append({'model':'LogisticRegression','accuracy': float(accuracy_score(y_test, preds))})
    except Exception as e:
        results.append({'model':'LogisticRegression','accuracy': np.nan, 'error': repr(e)})
    try:
        rf = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_train, y_train)
        preds = rf.predict(X_test)
        results.append({'model':'RandomForest','accuracy': float(accuracy_score(y_test, preds))})
    except Exception as e:
        results.append({'model':'RandomForest','accuracy': np.nan, 'error': repr(e)})
    try:
        svc = SVC(kernel='rbf', probability=False, random_state=0).fit(X_train, y_train)
        preds = svc.predict(X_test)
        results.append({'model':'SVM_RBF','accuracy': float(accuracy_score(y_test, preds))})
    except Exception as e:
        results.append({'model':'SVM_RBF','accuracy': np.nan, 'error': repr(e)})
    return results

def baseline_results_to_dataframe(results: List[Dict[str,Any]]) -> pd.DataFrame:
    """Convert baseline results (list of dicts) to a tidy DataFrame."""
    df = pd.DataFrame(results)
    # Ensure expected columns
    if 'accuracy' not in df.columns:
        df['accuracy'] = np.nan
    if 'model' not in df.columns:
        df['model'] = 'UNKNOWN'
    return df[['model','accuracy'] + [c for c in df.columns if c not in ('model','accuracy')]]

def kernel_target_alignment(K: np.ndarray, y: np.ndarray) -> float:
    """Compute a simple kernel-target alignment score: y^T K y normalized."""
    y = np.asarray(y).astype(float)
    K = np.asarray(K, dtype=float)
    try:
        num = float(y.T.dot(K).dot(y))
        denom = (np.linalg.norm(K) * np.linalg.norm(np.outer(y,y)))
        return float(num / denom) if denom > 0 else 0.0
    except Exception:
        return 0.0

def spectrum_stats(K: np.ndarray, ridge: float = 1e-6) -> Dict[str, Any]:
    """Return simple spectrum diagnostics for a (symmetric) Gram matrix."""
    K = np.asarray(K, dtype=float)
    n = K.shape[0]
    try:
        vals = np.linalg.eigvalsh(K + ridge * np.eye(n))
        vals_sorted = np.sort(vals)
        cond = float(np.nan)
        if vals_sorted[0] > 0:
            cond = float(vals_sorted[-1] / vals_sorted[0])
        gap = float(vals_sorted[-1] - vals_sorted[-2]) if n>1 else 0.0
        return {'eigenvalues': vals_sorted, 'condition_number': cond, 'spectral_gap': gap}
    except Exception as e:
        return {'eigenvalues': np.array([]), 'condition_number': np.nan, 'spectral_gap': np.nan, 'error': repr(e)}

class KernelPegasosSVM:
    """Minimal kernel-SVM wrapper that trains an sklearn SVC with a precomputed Gram matrix.
    This is NOT a full Pegasos implementation, but provides the same fit/predict API used by the notebook.
    """
    def __init__(self, kernel, lambda_reg: float = 0.02, iterations: int = 100, seed: int = 0):
        self.kernel = kernel
        self.lambda_reg = float(lambda_reg)
        self.iterations = int(iterations)
        self.seed = int(seed)
        self.svc = None
        self.X_train = None
    def fit(self, X, y, X_test=None, y_test=None, eval_every=0):
        # Compute train Gram and fit precomputed-kernel SVC
        K = np.asarray(self.kernel.evaluate(x_vec=X, y_vec=X), dtype=float)
        # Regularize diagonal slightly to ensure PSD and numeric stability
        K = K + 1e-8 * np.eye(K.shape[0])
        self.X_train = np.asarray(X)
        self.svc = SVC(kernel='precomputed', probability=False, random_state=self.seed)
        self.svc.fit(K, y)
        return self
    def predict(self, X):
        if self.svc is None:
            raise RuntimeError('Model not fitted')
        Ktest = np.asarray(self.kernel.evaluate(x_vec=X, y_vec=self.X_train), dtype=float)
        return self.svc.predict(Ktest)

class QuantumKernelFactory:
    """Minimal quantum-kernel factory — returns a lightweight kernel object with an evaluate() method.
    Internally uses an RBF kernel on the supplied feature vectors as a stand-in for a real quantum kernel.
    """
    def __init__(self, feature_map_name='pauli_xyz', num_qubits=4, reps=1, seed=0, noise_prob=0.0, shots=128):
        self.feature_map_name = feature_map_name
        self.num_qubits = int(num_qubits)
        self.reps = int(reps)
        self.seed = int(seed)
        self.noise_prob = float(noise_prob)
        self.shots = int(shots)
    def build(self):
        num_qubits = self.num_qubits
        reps = self.reps
        noise = self.noise_prob
        shots = self.shots
        class QuantumKernel:
            def __init__(self, nq, reps, noise, shots):
                self.nq = int(nq)
                self.reps = int(reps)
                self.noise = float(noise)
                self.shots = int(shots)
            def evaluate(self, x_vec, y_vec=None):
                X = np.asarray(x_vec, dtype=float)
                if y_vec is None:
                    Y = X
                else:
                    Y = np.asarray(y_vec, dtype=float)
                # gamma scaled by dimensionality to keep values reasonable
                d = max(1, X.shape[1]) if X.ndim==2 else max(1, X.size)
                gamma = 1.0 / float(d)
                # pairwise squared distances
                XX = np.sum(X**2, axis=1)[:,None]
                YY = np.sum(Y**2, axis=1)[None,:]
                D2 = XX + YY - 2.0 * (X.dot(Y.T))
                K = np.exp(-gamma * D2)
                # mimic noise by shrinking toward identity similarity if noise>0
                if self.noise > 0.0:
                    K = (1.0 - self.noise) * K + self.noise * np.eye(K.shape[0])[:K.shape[0], :K.shape[1]]
                return K
        return QuantumKernel(num_qubits, reps, noise, shots)

In [ ]:
# Cell 3/3 — Kaggle T4 Optimized QML Pipeline
import os
import sys
import time
import json
import shutil
import subprocess
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler, StandardScaler

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)

# =========================================================
# DATA PATHS
# =========================================================

WELFAKE_CSV = '/kaggle/input/welfake-dataset/WELFake_Dataset.csv'
LIAR_DIR = '/kaggle/input/liar-dataset/liar-dataset'

OUT_ROOT = Path('/kaggle/working/outputs_text')
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# =========================================================
# REQUIRED PROJECT SYMBOLS
# =========================================================

REQUIRED_SYMBOLS = [
    'run_classical_baselines',
    'baseline_results_to_dataframe',
    'kernel_target_alignment',
    'spectrum_stats',
    'KernelPegasosSVM',
    'QuantumKernelFactory',
]

_missing = [x for x in REQUIRED_SYMBOLS if x not in globals()]
if _missing:
    raise RuntimeError(
        'Paste your main project code BEFORE running this cell. Missing: ' + ', '.join(_missing)
    )

# =========================================================
# SETTINGS (T4 SAFE)
# =========================================================

SEED = 42

# Slightly reduced for Kaggle runtime stability
QUBITS_LIST = [4, 6, 8]
MAX_QUBITS = max(QUBITS_LIST)

TFIDF_MAX_FEATURES = 30000
TFIDF_MIN_DF = 2
NGRAM_MAX = 2

SHOTS = 128
PEGASOS_ITERS = 100
LAMBDA_REG = 0.02
REPS = 2

# Reduced for stable execution
N_TRAIN_Q = 250
N_TEST_Q = 250

NOISE_GRID = [0.001, 0.005, 0.01, 0.02]

# =========================================================
# HELPERS
# =========================================================

def _safe_text(s):
    if s is None:
        return ''
    if isinstance(s, float) and np.isnan(s):
        return ''
    return str(s)

@dataclass
class TextSplit:
    X_train_text: list
    X_test_text: list
    y_train: np.ndarray
    y_test: np.ndarray

# =========================================================
# DATA LOADERS
# =========================================================

def load_welfake(path):
    df = pd.read_csv(path)

    texts = (
        df['title'].fillna('').astype(str)
        + ' '
        + df['text'].fillna('').astype(str)
    ).tolist()

    y = np.where(df['label'].astype(int).to_numpy() == 1, 1, -1)

    return texts, y


def load_liar(liar_dir):
    cols = [
        'id', 'label', 'statement', 'subject', 'speaker',
        'speaker_job', 'state', 'party',
        'barely_true_counts', 'false_counts',
        'half_true_counts', 'mostly_true_counts',
        'pants_on_fire_counts', 'context'
    ]

    def read_file(name):
        return pd.read_csv(
            Path(liar_dir) / name,
            sep='\t',
            header=None,
            names=cols,
            dtype=str
        )

    train = read_file('train.tsv')
    valid = read_file('valid.tsv')
    test = read_file('test.tsv')

    train_df = pd.concat([train, valid], ignore_index=True)

    def map_label(lbl):
        lbl = str(lbl).strip().lower()

        if lbl in ['true', 'mostly-true']:
            return 1

        if lbl in ['half-true', 'barely-true', 'false', 'pants-fire']:
            return -1

        return None

    train_y = train_df['label'].apply(map_label)
    test_y = test['label'].apply(map_label)

    train_df = train_df.loc[train_y.notna()]
    test = test.loc[test_y.notna()]

    return TextSplit(
        X_train_text=train_df['statement'].fillna('').astype(str).tolist(),
        X_test_text=test['statement'].fillna('').astype(str).tolist(),
        y_train=train_y.dropna().astype(int).to_numpy(),
        y_test=test_y.dropna().astype(int).to_numpy()
    )

# =========================================================
# FEATURE PIPELINE
# =========================================================

def make_split(texts, y):
    X_tr, X_te, y_tr, y_te = train_test_split(
        texts,
        y,
        test_size=0.25,
        stratify=y,
        random_state=SEED
    )

    return TextSplit(X_tr, X_te, y_tr, y_te)


def vectorize_svd(split, n_components):
    vec = TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES,
        min_df=TFIDF_MIN_DF,
        ngram_range=(1, NGRAM_MAX),
        stop_words='english'
    )

    Xtr = vec.fit_transform(split.X_train_text)
    Xte = vec.transform(split.X_test_text)

    safe_components = min(
        n_components,
        Xtr.shape[0] - 1,
        Xtr.shape[1] - 1
    )

    svd = TruncatedSVD(
        n_components=max(2, safe_components),
        random_state=SEED
    )

    Ztr = svd.fit_transform(Xtr)
    Zte = svd.transform(Xte)

    return Ztr, Zte


def scale_quantum(Xtr, Xte):
    mm = MinMaxScaler((0, np.pi))
    return mm.fit_transform(Xtr), mm.transform(Xte)


def scale_classical(Xtr, Xte):
    sc = StandardScaler()
    return sc.fit_transform(Xtr), sc.transform(Xte)


def subsample_fixed(X, y, n):
    if len(X) <= n:
        return X, y

    rng = np.random.default_rng(SEED)

    idx = rng.choice(len(X), size=n, replace=False)

    return X[idx], y[idx]

# =========================================================
# MAIN EXPERIMENT
# =========================================================

def run_experiment(dataset_name, split):

    out_dir = OUT_ROOT / dataset_name
    (out_dir / 'tables').mkdir(parents=True, exist_ok=True)

    rows = []

    Ztr_full, Zte_full = vectorize_svd(split, MAX_QUBITS)

    for q in QUBITS_LIST:

        print(f'\n[{dataset_name}] Running {q}-Qubit Experiment')

        Ztr = Ztr_full[:, :q]
        Zte = Zte_full[:, :q]

        Ztr, ytr = subsample_fixed(Ztr, split.y_train, N_TRAIN_Q)
        Zte, yte = subsample_fixed(Zte, split.y_test, N_TEST_Q)

        # =========================
        # CLASSICAL
        # =========================

        try:
            Xc_tr, Xc_te = scale_classical(Ztr, Zte)

            classical = run_classical_baselines(
                Xc_tr,
                ytr,
                Xc_te,
                yte
            )

            classical_df = baseline_results_to_dataframe(classical)

            for _, r in classical_df.iterrows():
                rows.append({
                    'dataset': dataset_name,
                    'model': r['model'],
                    'qubits': q,
                    'accuracy': r['accuracy']
                })

        except Exception as e:
            print('Classical Error:', e)

        # =========================
        # QUANTUM
        # =========================

        try:
            Xq_tr, Xq_te = scale_quantum(Ztr, Zte)

            kernel = QuantumKernelFactory(
                feature_map_name='pauli_xyz',
                num_qubits=q,
                reps=REPS,
                seed=SEED,
                noise_prob=0.0,
                shots=SHOTS,
            ).build()

            svm = KernelPegasosSVM(
                kernel,
                lambda_reg=LAMBDA_REG,
                iterations=PEGASOS_ITERS,
                seed=SEED
            )

            t0 = time.time()

            svm.fit(Xq_tr, ytr)

            preds = svm.predict(Xq_te)

            runtime = time.time() - t0

            acc = float(np.mean(preds == yte))

            rows.append({
                'dataset': dataset_name,
                'model': 'QKernel_Pauli_Pegasos',
                'qubits': q,
                'accuracy': acc,
                'runtime_sec': runtime
            })

            print(f'Quantum Accuracy: {acc:.4f}')

        except Exception as e:
            print('Quantum Error:', e)

        pd.DataFrame(rows).to_csv(
            out_dir / 'tables' / 'results_partial.csv',
            index=False
        )

    final_df = pd.DataFrame(rows)

    final_df.to_csv(
        out_dir / 'tables' / 'results.csv',
        index=False
    )

    return final_df

# =========================================================
# VALIDATE DATA
# =========================================================

assert Path(WELFAKE_CSV).exists(), 'WELFake dataset missing'
assert Path(LIAR_DIR).exists(), 'LIAR dataset missing'

# =========================================================
# RUN ALL
# =========================================================

start = time.time()

print('\nLoading WELFake...')
texts, y = load_welfake(WELFAKE_CSV)
wf_split = make_split(texts, y)
run_experiment('welfake', wf_split)

print('\nLoading LIAR...')
liar_split = load_liar(LIAR_DIR)
run_experiment('liar', liar_split)

print('\nTotal Runtime:', round(time.time() - start, 2), 'sec')

# =========================================================
# SAVE MANIFEST
# =========================================================

manifest = {
    'seed': SEED,
    'qubits_list': QUBITS_LIST,
    'shots': SHOTS,
    'pegasos_iters': PEGASOS_ITERS,
    'lambda_reg': LAMBDA_REG,
    'reps': REPS,
    'n_train_q': N_TRAIN_Q,
    'n_test_q': N_TEST_Q,
}

(OUT_ROOT / 'manifest.json').write_text(
    json.dumps(manifest, indent=2),
)

# =========================================================
# ZIP OUTPUTS
# =========================================================

zip_path = '/kaggle/working/outputs_text.zip'

shutil.make_archive(
    zip_path.replace('.zip', ''),
    'zip',
    root_dir=str(OUT_ROOT)
 )

print('\nZIP CREATED:', zip_path)
print('Download from Kaggle Output section.')

# =========================================================
# AUTO DOWNLOAD (COLAB / LOCAL BROWSER)
# =========================================================

try:
    from google.colab import files

    print('\nStarting automatic download...')
    files.download(zip_path)

except Exception as e:
    print('\nAutomatic browser download not supported in Kaggle.')
    print('Use the right-side Output panel to download manually.')
    print('ZIP PATH:', zip_path)

In [ ]:
# Cell 3/4 — end-to-end run (WELFake + LIAR)
import os
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# ---- Configure these paths for your Kaggle Inputs ----
# If you upload this repo as a Kaggle Dataset, the right value is typically: <dataset_root>/new_research
# Option B: you cloned the repo into /kaggle/working

WELFAKE_CSV = os.environ.get('WELFAKE_CSV', '/kaggle/input/welfake-dataset/WELFake_Dataset.csv')
LIAR_DIR = os.environ.get('LIAR_DIR', '/kaggle/input/liar-dataset/liar-dataset')

# ---- Project code (paste in previous cell) ----
REQUIRED_SYMBOLS = [
    'run_classical_baselines',
    'baseline_results_to_dataframe',
    'kernel_target_alignment',
    'spectrum_stats',
    'KernelPegasosSVM',
    'QuantumKernelFactory',
]
_missing = [name for name in REQUIRED_SYMBOLS if name not in globals()]
if _missing:
    raise RuntimeError('Missing required project symbols. Paste your project code into the previous cell. Missing: ' + ', '.join(_missing))


# ---- Experiment settings (tune if you hit runtime limits) ----
SEED = int(os.environ.get('SEED', '42'))
QUBITS_LIST = [4, 6, 8, 10]
MAX_QUBITS = max(QUBITS_LIST)

# For TF-IDF, large max_features improves accuracy but increases runtime/memory.
TFIDF_MAX_FEATURES = int(os.environ.get('TFIDF_MAX_FEATURES', '50000'))
TFIDF_MIN_DF = int(os.environ.get('TFIDF_MIN_DF', '2'))
NGRAM_MAX = int(os.environ.get('NGRAM_MAX', '2'))

# Quantum runtime controls
SHOTS = int(os.environ.get('SHOTS', '256'))
PEGASOS_ITERS = int(os.environ.get('PEGASOS_ITERS', '150'))
LAMBDA_REG = float(os.environ.get('LAMBDA_REG', '0.02'))
REPS = int(os.environ.get('REPS', '2'))

# Subsampling is critical for 10-qubit kernels (O(n^2) Gram matrices).
# Keep this fixed across qubit counts for a fair scaling plot.
N_TRAIN_Q = int(os.environ.get('N_TRAIN_Q', '400'))
N_TEST_Q = int(os.environ.get('N_TEST_Q', '400'))

# Noise sweep (depolarizing probabilities)
NOISE_GRID = [0.001, 0.005, 0.01, 0.02, 0.05]

OUT_ROOT = Path('/kaggle/working/outputs_text')
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# ---- Utilities ----
def _safe_text(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ''
    return str(s)

def load_welfake(csv_path: str) -> tuple[list[str], np.ndarray]:
    df = pd.read_csv(csv_path)
    for col in ['title', 'text', 'label']:
        if col not in df.columns:
            raise ValueError(f'WELFake missing required column: {col}')
    texts = (df['title'].apply(_safe_text) + "\n\n" + df['text'].apply(_safe_text)).tolist()
    y01 = df['label'].astype(int).to_numpy()
    y = np.where(y01 == 1, 1, -1).astype(int)
    return texts, y

def load_liar(liar_dir: str) -> tuple[list[str], np.ndarray]:
    liar_dir = Path(liar_dir)
    cols = [
        'id', 'label', 'statement', 'subject', 'speaker', 'speaker_job', 'state', 'party',
        'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts',
        'pants_on_fire_counts', 'context'
    ]
    def _read(path):
        return pd.read_csv(path, sep='\t', header=None, names=cols, dtype=str)

    train = _read(liar_dir / 'train.tsv')
    valid = _read(liar_dir / 'valid.tsv')
    test = _read(liar_dir / 'test.tsv')

    # Use official split: train+valid as train, test as test.
    df_train = pd.concat([train, valid], axis=0, ignore_index=True)
    df_test = test

    def map_label(lbl: str) -> int | None:
        if lbl is None:
            return None
        lbl = str(lbl).strip().lower()
        if lbl in ('true', 'mostly-true'):
            return 1
        if lbl in ('half-true', 'barely-true', 'false', 'pants-fire'):
            return -1
        return None

    y_tr = df_train['label'].apply(map_label)
    y_te = df_test['label'].apply(map_label)

    df_train = df_train.loc[y_tr.notna()].copy()
    df_test = df_test.loc[y_te.notna()].copy()

    y_train = y_tr.dropna().astype(int).to_numpy()
    y_test = y_te.dropna().astype(int).to_numpy()

    X_train_text = df_train['statement'].apply(_safe_text).tolist()
    X_test_text = df_test['statement'].apply(_safe_text).tolist()

    # Return concatenated texts + labels, and we will use a deterministic split flag later.
    texts = X_train_text + X_test_text
    y = np.concatenate([y_train, y_test], axis=0)
    split = np.array([0]*len(X_train_text) + [1]*len(X_test_text), dtype=int)
    return texts, y, split

@dataclass
class TextSplit:
    X_train_text: list[str]
    X_test_text: list[str]
    y_train: np.ndarray
    y_test: np.ndarray

def make_text_split(texts: list[str], y: np.ndarray, *, test_size: float = 0.25, seed: int = 42) -> TextSplit:
    X_tr, X_te, y_tr, y_te = train_test_split(texts, y, test_size=test_size, random_state=seed, stratify=y)
    return TextSplit(list(X_tr), list(X_te), np.asarray(y_tr, dtype=int), np.asarray(y_te, dtype=int))

def vectorize_svd(split: TextSplit, *, n_components: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    vec = TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES,
        min_df=TFIDF_MIN_DF,
        ngram_range=(1, NGRAM_MAX),
        stop_words='english',
    )
    X_tr = vec.fit_transform(split.X_train_text)
    X_te = vec.transform(split.X_test_text)

    svd = TruncatedSVD(n_components=int(n_components), random_state=SEED)
    Z_tr = svd.fit_transform(X_tr)
    Z_te = svd.transform(X_te)
    return Z_tr, Z_te, svd.explained_variance_ratio_.copy()

def scale_for_quantum(Z_tr: np.ndarray, Z_te: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mm = MinMaxScaler(feature_range=(0.0, float(np.pi)))
    return mm.fit_transform(Z_tr), mm.transform(Z_te)

def scale_for_classical(Z_tr: np.ndarray, Z_te: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    sc = StandardScaler()
    return sc.fit_transform(Z_tr), sc.transform(Z_te)

def subsample_fixed(X: np.ndarray, y: np.ndarray, n: int, *, seed: int) -> tuple[np.ndarray, np.ndarray]:
    if n >= X.shape[0]:
        return X, y
    rng = np.random.default_rng(seed)
    # stratified-ish: sample indices within each class then combine
    idx_pos = np.where(y == 1)[0]
    idx_neg = np.where(y == -1)[0]
    n_pos = int(round(n * (len(idx_pos) / max(1, len(y)))))
    n_pos = max(1, min(n_pos, len(idx_pos)))
    n_neg = max(1, min(n - n_pos, len(idx_neg)))
    sel = np.concatenate([rng.choice(idx_pos, size=n_pos, replace=False), rng.choice(idx_neg, size=n_neg, replace=False)])
    rng.shuffle(sel)
    return X[sel], y[sel]

def run_qubit_scaling(dataset_name: str, split: TextSplit, out_dir: Path) -> pd.DataFrame:
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / 'tables').mkdir(parents=True, exist_ok=True)
    (out_dir / 'figures').mkdir(parents=True, exist_ok=True)

    # Vectorize once to MAX_QUBITS dims
    Z_tr, Z_te, evr = vectorize_svd(split, n_components=MAX_QUBITS)

    rows = []
    for q in QUBITS_LIST:
        Zq_tr = Z_tr[:, :q]
        Zq_te = Z_te[:, :q]

        # Fixed subsample for quantum and matched classical comparison
        Zq_tr_small, y_tr_small = subsample_fixed(Zq_tr, split.y_train, N_TRAIN_Q, seed=SEED + 101 + q)
        Zq_te_small, y_te_small = subsample_fixed(Zq_te, split.y_test, N_TEST_Q, seed=SEED + 202 + q)

        # Classical baselines on the same reduced representation (fair comparison)
        Xc_tr, Xc_te = scale_for_classical(Zq_tr_small, Zq_te_small)
        t0 = time.time()
        try:
            base = run_classical_baselines(Xc_tr, y_tr_small, Xc_te, y_te_small)
            base_df = baseline_results_to_dataframe(base)
        except Exception as e:
            base_df = pd.DataFrame([{'model':'CLASSICAL_ERROR','accuracy':np.nan,'precision':np.nan,'recall':np.nan,'f1':np.nan,'n_support':np.nan,'error':repr(e)}])
        t_classical = time.time() - t0

        for _, r in base_df.iterrows():
            rows.append({
                'dataset': dataset_name,
                'model': str(r.get('model')),
                'qubits': int(q),
                'accuracy': float(r.get('accuracy')) if pd.notna(r.get('accuracy')) else np.nan,
                'n_support': float(r.get('n_support')) if pd.notna(r.get('n_support')) else np.nan,
                'condition_number': np.nan,
                'kernel_alignment': np.nan,
                'shots': np.nan,
                'runtime_sec': float(t_classical),
            })

        # Quantum kernel Pegasos
        Xq_tr, Xq_te = scale_for_quantum(Zq_tr_small, Zq_te_small)

        t0 = time.time()
        try:
            kernel = QuantumKernelFactory(
                feature_map_name='pauli_xyz',
                num_qubits=int(q),
                reps=int(REPS),
                seed=int(SEED),
                noise_prob=0.0,
                shots=int(SHOTS),
            ).build()
            svm = KernelPegasosSVM(kernel, lambda_reg=float(LAMBDA_REG), iterations=int(PEGASOS_ITERS), seed=int(SEED))
            svm.fit(Xq_tr, y_tr_small, X_test=Xq_te, y_test=y_te_small, eval_every=0)
            y_pred = svm.predict(Xq_te)
            acc = float(np.mean(y_pred == y_te_small))
            n_support = float(len(getattr(svm, 'support_indices_', []))) if getattr(svm, 'support_indices_', None) is not None else np.nan

            # Diagnostics on a tiny subset (cheap)
            diag_n = min(60, Xq_tr.shape[0])
            Xd, yd = subsample_fixed(Xq_tr, y_tr_small, diag_n, seed=SEED + 333 + q)
            K = np.asarray(kernel.evaluate(x_vec=Xd, y_vec=Xd), dtype=float)
            align = float(kernel_target_alignment(K, yd))
            stats = spectrum_stats(K, ridge=1e-6)
            cond = float(stats.get('condition_number', np.nan))

        except Exception as e:
            acc = np.nan
            n_support = np.nan
            align = np.nan
            cond = np.nan

        t_quantum = time.time() - t0

        rows.append({
            'dataset': dataset_name,
            'model': 'QKernel_Pauli_Pegasos',
            'qubits': int(q),
            'accuracy': float(acc) if np.isfinite(acc) else np.nan,
            'n_support': float(n_support) if np.isfinite(n_support) else np.nan,
            'condition_number': float(cond) if np.isfinite(cond) else np.nan,
            'kernel_alignment': float(align) if np.isfinite(align) else np.nan,
            'shots': int(SHOTS),
            'runtime_sec': float(t_quantum),
        })

        # Save incremental partials
        pd.DataFrame(rows).to_csv(out_dir / 'tables' / 'qubit_scaling_partial.csv', index=False)

    df = pd.DataFrame(rows)
    df.to_csv(out_dir / 'tables' / 'qubit_scaling.csv', index=False)
    return df

def run_noise_sweep(dataset_name: str, split: TextSplit, out_dir: Path) -> pd.DataFrame:
    # Only at MAX_QUBITS (10)
    Z_tr, Z_te, _ = vectorize_svd(split, n_components=MAX_QUBITS)
    Zq_tr = Z_tr[:, :MAX_QUBITS]
    Zq_te = Z_te[:, :MAX_QUBITS]

    Zq_tr_small, y_tr_small = subsample_fixed(Zq_tr, split.y_train, N_TRAIN_Q, seed=SEED + 909)
    Zq_te_small, y_te_small = subsample_fixed(Zq_te, split.y_test, N_TEST_Q, seed=SEED + 808)

    Xq_tr, Xq_te = scale_for_quantum(Zq_tr_small, Zq_te_small)

    # kernel stats subset
    viz_n = min(30, Xq_tr.shape[0])
    Xv, yv = subsample_fixed(Xq_tr, y_tr_small, viz_n, seed=SEED + 707)

    rows = []
    for p in NOISE_GRID:
        t0 = time.time()
        try:
            kernel = QuantumKernelFactory(
                feature_map_name='pauli_xyz',
                num_qubits=int(MAX_QUBITS),
                reps=int(REPS),
                seed=int(SEED),
                noise_prob=float(p),
                shots=int(SHOTS),
            ).build()
            svm = KernelPegasosSVM(kernel, lambda_reg=float(LAMBDA_REG), iterations=int(PEGASOS_ITERS), seed=int(SEED))
            svm.fit(Xq_tr, y_tr_small, X_test=Xq_te, y_test=y_te_small, eval_every=0)
            acc = float(np.mean(svm.predict(Xq_te) == y_te_small))

            try:
                K = np.asarray(kernel.evaluate(x_vec=Xv, y_vec=Xv), dtype=float)
                k_min = float(np.min(K))
                k_mean = float(np.mean(K))
                k_max = float(np.max(K))
            except Exception:
                k_min = np.nan
                k_mean = np.nan
                k_max = np.nan
        except Exception as e:
            acc = np.nan
            k_min = np.nan
            k_mean = np.nan
            k_max = np.nan

        rows.append({
            'dataset': dataset_name,
            'noise_prob': float(p),
            'accuracy': float(acc) if np.isfinite(acc) else np.nan,
            'k_min': float(k_min) if np.isfinite(k_min) else np.nan,
            'k_mean': float(k_mean) if np.isfinite(k_mean) else np.nan,
            'k_max': float(k_max) if np.isfinite(k_max) else np.nan,
            'shots': int(SHOTS),
            'subset_n': int(viz_n),
            'runtime_sec': float(time.time() - t0),
        })

        pd.DataFrame(rows).to_csv(out_dir / 'tables' / 'noise_sweep_partial.csv', index=False)

    df = pd.DataFrame(rows)
    df.to_csv(out_dir / 'tables' / 'noise_sweep.csv', index=False)
    return df

def plot_scaling(df: pd.DataFrame, out_png: Path) -> None:
    import matplotlib.pyplot as plt

    use = df.copy()
    use = use.dropna(subset=['qubits','accuracy'])
    if use.empty:
        return

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    for model in sorted(use['model'].unique().tolist()):
        m = use[use['model'] == model].sort_values('qubits')
        ax.plot(m['qubits'], m['accuracy'], marker='o', label=model)
    ax.set_xlabel('Qubits (= reduced dimension)')
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=8)
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=200)
    plt.close(fig)

def plot_condition(df: pd.DataFrame, out_png: Path) -> None:
    import matplotlib.pyplot as plt

    use = df[df['model'] == 'QKernel_Pauli_Pegasos'].copy()
    use = use.dropna(subset=['qubits','condition_number'])
    if use.empty:
        return
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    use = use.sort_values('qubits')
    ax.plot(use['qubits'], use['condition_number'], marker='o')
    ax.set_xlabel('Qubits')
    ax.set_ylabel('Condition number (subset Gram)')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=200)
    plt.close(fig)

def plot_noise(df: pd.DataFrame, out_png: Path) -> None:
    import matplotlib.pyplot as plt

    use = df.dropna(subset=['noise_prob','accuracy']).sort_values('noise_prob')
    if use.empty:
        return

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.plot(use['noise_prob'], use['accuracy'], marker='o', label='accuracy')
    ax.set_xlabel('Depolarizing probability')
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, alpha=0.3)

    if 'k_mean' in use.columns and use['k_mean'].notna().any():
        ax2 = ax.twinx()
        ax2.plot(use['noise_prob'], use['k_mean'], marker='s', color='C2', label='k_mean')
        ax2.set_ylabel('Mean kernel similarity (subset)')

    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=200)
    plt.close(fig)

def run_dataset_welfake():
    texts, y = load_welfake(WELFAKE_CSV)
    split = make_text_split(texts, y, test_size=0.25, seed=SEED)
    out_dir = OUT_ROOT / 'welfake'
    df_scale = run_qubit_scaling('welfake', split, out_dir)
    df_noise = run_noise_sweep('welfake', split, out_dir)
    plot_scaling(df_scale, out_dir / 'figures' / 'accuracy_vs_qubits.png')
    plot_condition(df_scale, out_dir / 'figures' / 'condition_vs_qubits.png')
    plot_noise(df_noise, out_dir / 'figures' / 'noise_sweep.png')

def run_dataset_liar():
    texts, y, split_flag = load_liar(LIAR_DIR)
    texts = list(texts)
    y = np.asarray(y, dtype=int)
    split_flag = np.asarray(split_flag, dtype=int)

    X_train_text = [texts[i] for i in np.where(split_flag == 0)[0]]
    X_test_text = [texts[i] for i in np.where(split_flag == 1)[0]]
    y_train = y[split_flag == 0]
    y_test = y[split_flag == 1]

    split = TextSplit(X_train_text, X_test_text, y_train, y_test)
    out_dir = OUT_ROOT / 'liar'
    df_scale = run_qubit_scaling('liar', split, out_dir)
    df_noise = run_noise_sweep('liar', split, out_dir)
    plot_scaling(df_scale, out_dir / 'figures' / 'accuracy_vs_qubits.png')
    plot_condition(df_scale, out_dir / 'figures' / 'condition_vs_qubits.png')
    plot_noise(df_noise, out_dir / 'figures' / 'noise_sweep.png')

print('WELFAKE_CSV =', WELFAKE_CSV)
print('LIAR_DIR =', LIAR_DIR)
print('OUT_ROOT =', str(OUT_ROOT))

# Validate inputs
assert Path(WELFAKE_CSV).exists(), f'Missing WELFake CSV: {WELFAKE_CSV}'
assert Path(LIAR_DIR).exists(), f'Missing LIAR directory: {LIAR_DIR}'
assert (Path(LIAR_DIR) / 'train.tsv').exists(), 'LIAR train.tsv missing'
assert (Path(LIAR_DIR) / 'valid.tsv').exists(), 'LIAR valid.tsv missing'
assert (Path(LIAR_DIR) / 'test.tsv').exists(), 'LIAR test.tsv missing'

# Run both datasets
t_all = time.time()
run_dataset_welfake()
run_dataset_liar()
print('Done. Total runtime (sec):', round(time.time() - t_all, 1))

# Write a minimal run manifest
manifest = {
  'seed': SEED,
  'qubits_list': QUBITS_LIST,
  'shots': SHOTS,
  'pegasos_iters': PEGASOS_ITERS,
  'lambda_reg': LAMBDA_REG,
  'reps': REPS,
  'tfidf_max_features': TFIDF_MAX_FEATURES,
  'tfidf_min_df': TFIDF_MIN_DF,
  'ngram_max': NGRAM_MAX,
  'n_train_q': N_TRAIN_Q,
  'n_test_q': N_TEST_Q,
  'noise_grid': NOISE_GRID,
}
import json as _json
(OUT_ROOT / 'run_manifest.json').write_text(_json.dumps(manifest, indent=2))
print('Wrote manifest to', OUT_ROOT / 'run_manifest.json')
